# 06 — Proactive Audio: Controlling When the Model Speaks

## What Is Proactive Audio?

In a standard turn-based conversation the model waits until the user finishes speaking
before replying. **Proactive audio** flips this: the model can speak without waiting,
respond to intentional silence, or even greet the user the moment a session opens.

### Two Activity-Detection Modes

| Mode | How it works | Best for |
|------|-------------|----------|
| **Automatic (server-side VAD)** | Server detects speech start/end in the audio stream automatically | General use, quick prototypes |
| **Manual** | Client sends explicit `ActivityStart` / `ActivityEnd` signals | Call centers, kiosks, precise PTT control |

### Use Cases
- **Call-center IVR**: greet callers immediately; don't wait for them to speak first  
- **Kiosk / info-booth**: proactive introduction when a user approaches  
- **Push-to-talk devices**: manual control over when audio is interpreted  
- **Noise-heavy environments**: tune sensitivity so background noise doesn't trigger responses

---

### Activity Detection — ASCII Diagram

```
AUTOMATIC MODE (server-side VAD)
=================================
Audio stream →  [silence]  [SPEECH..........]  [silence]
                              ↑                       ↑
                     VAD detects start         VAD detects end
                     (internally)              → model responds

MANUAL MODE (client-controlled)
================================
Client sends:  ActivityStart → [audio chunks] → ActivityEnd
                    ↑                                  ↑
           YOU signal start                   YOU signal end
                                              → model responds
```

---

**Notebook structure**
1. Setup  
2. Demo 1 — Automatic VAD (default)  
3. Demo 2 — Manual activity detection  
4. Demo 3 — Silence sensitivity tuning  
5. Demo 4 — Proactive speaking (model speaks first)  
6. Key takeaways

## Setup

Install the google-genai SDK (skip if already installed).

In [ ]:
# Install / upgrade google-genai
!pip install -q google-genai numpy

In [ ]:
import nest_asyncio; nest_asyncio.apply()   # allow asyncio.run() inside Jupyter

import asyncio
import os
import json
import numpy as np
import IPython.display as ipd
from google import genai
from google.genai import types

# ── credentials ──────────────────────────────────────────────────────────────
from dotenv import load_dotenv
load_dotenv()  # loads GEMINI_API_KEY from .env

API_KEY = os.environ.get("GEMINI_API_KEY", "")
MODEL   = "gemini-3.1-flash-live-preview"   # Live API model

client = genai.Client(api_key=API_KEY)
print(f"Client ready. Model: {MODEL}")

In [ ]:
# ── audio helpers ─────────────────────────────────────────────────────────────

def make_pcm(duration: float = 2.0, rate: int = 16000, freq: float = 440.0) -> bytes:
    """Generate a synthetic sine-wave PCM burst (simulates recorded speech)."""
    t = np.linspace(0, duration, int(rate * duration))
    wave = (np.sin(2 * np.pi * freq * t) * 0.3 * 32767).astype(np.int16)
    return wave.tobytes()


def play_pcm(raw_bytes: bytes, rate: int = 24000) -> ipd.Audio:
    """Wrap raw PCM bytes in an IPython Audio widget (model output is 24 kHz)."""
    arr = np.frombuffer(raw_bytes, dtype=np.int16).astype(np.float32) / 32768.0
    return ipd.Audio(arr, rate=rate, autoplay=False)


async def collect_response(session, label: str = "") -> dict:
    """
    Drain the session receive() loop until turn_complete or go_away.
    Returns {'audio': bytes, 'transcript': str}.
    """
    audio_chunks = []
    transcript_parts = []

    async for resp in session.receive():
        # Raw audio data
        if resp.data:
            audio_chunks.append(resp.data)

        sc = resp.server_content
        if sc:
            # Output transcription
            if sc.output_transcription and sc.output_transcription.text:
                transcript_parts.append(sc.output_transcription.text)

            # Text modality (when response_modalities includes TEXT)
            if sc.model_turn:
                for part in sc.model_turn.parts:
                    if hasattr(part, "text") and part.text:
                        transcript_parts.append(part.text)

            if sc.turn_complete:
                print(f"[{label}] Turn complete.")
                break
            if sc.interrupted:
                print(f"[{label}] Interrupted by server.")
                break

        if resp.go_away:
            print(f"[{label}] GoAway received — session ending soon.")
            break

    return {
        "audio": b"".join(audio_chunks),
        "transcript": "".join(transcript_parts),
    }

print("Helpers defined.")

---
## Demo 1 — Automatic Activity Detection (Server-Side VAD)

This is the **default** behaviour. The server listens to your audio stream and
automatically decides when speech starts and ends. You do not need to configure
anything special — just open a session and send audio.

Here we send a text message instead of real microphone audio so the demo runs
without a microphone. The turn-detection behaviour is the same.

In [ ]:
async def demo_automatic_vad():
    """Default config — server handles turn detection automatically."""

    # Minimal config: just ask for audio output + output transcription
    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        output_audio_transcription=types.AudioTranscriptionConfig(),
        system_instruction="You are a helpful assistant. Answer concisely in one sentence.",
    )

    async with client.aio.live.connect(model=MODEL, config=config) as session:
        print("Session open — automatic VAD is active.")

        # Send a text turn (equivalent to user speaking)
        await session.send_realtime_input(text="What is the capital of France?")

        result = await collect_response(session, label="AutoVAD")

    return result


result1 = asyncio.run(demo_automatic_vad())
print("\nTranscript:", result1["transcript"])

if result1["audio"]:
    print(f"Audio received: {len(result1['audio']):,} bytes")
    display(play_pcm(result1["audio"]))
else:
    print("(No audio bytes returned — check response_modalities)")

---
## Demo 2 — Manual Activity Detection

When you **disable** automatic activity detection you take full control:
- `ActivityStart` — tells the server "the user is now speaking"
- `ActivityEnd`   — tells the server "the user has finished; please respond"

This is essential for **push-to-talk** interfaces, where a physical button
determines when audio should be captured.

### Config key:
```python
realtime_input_config=types.RealtimeInputConfig(
    automatic_activity_detection=types.AutomaticActivityDetection(disabled=True),
    turn_coverage="TURN_INCLUDES_ONLY_ACTIVITY",
)
```

`TURN_INCLUDES_ONLY_ACTIVITY` means only audio between ActivityStart/ActivityEnd
is included in the model's context — idle audio before/after is ignored.

In [ ]:
async def demo_manual_activity():
    """Manual PTT-style activity detection."""

    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        realtime_input_config=types.RealtimeInputConfig(
            # Disable server-side VAD — WE control when speech starts/ends
            automatic_activity_detection=types.AutomaticActivityDetection(
                disabled=True
            ),
            # Only include audio that falls between ActivityStart and ActivityEnd
            turn_coverage="TURN_INCLUDES_ONLY_ACTIVITY",
        ),
        output_audio_transcription=types.AudioTranscriptionConfig(),
        system_instruction="You are a helpful assistant. Keep replies to one sentence.",
    )

    # Simulate a 1-second audio clip of the user asking a question
    # In production this would be real microphone PCM captured while PTT is held
    user_audio = make_pcm(duration=1.0, rate=16000, freq=300)

    async with client.aio.live.connect(model=MODEL, config=config) as session:
        print("Session open — manual activity detection enabled.")

        # ── Step 1: Signal that the user started speaking ──────────────────
        print(">>> Sending ActivityStart  (button pressed)")
        await session.send_realtime_input(
            activity_start=types.ActivityStart()
        )

        # ── Step 2: Send audio chunks (simulated) ─────────────────────────
        # In practice you stream mic chunks here in a loop
        await session.send_realtime_input(
            audio=types.Blob(data=user_audio, mime_type="audio/pcm;rate=16000")
        )
        print(">>> Audio chunk sent")

        # ── Step 3: Signal that the user stopped speaking ──────────────────
        print(">>> Sending ActivityEnd    (button released)")
        await session.send_realtime_input(
            activity_end=types.ActivityEnd()
        )

        # ── Step 4: Collect model response ────────────────────────────────
        result = await collect_response(session, label="ManualVAD")

    return result


result2 = asyncio.run(demo_manual_activity())
print("\nTranscript:", result2["transcript"] or "(audio-only response)")

if result2["audio"]:
    print(f"Audio received: {len(result2['audio']):,} bytes")
    display(play_pcm(result2["audio"]))

---
## Demo 3 — Silence Sensitivity Tuning

When using **automatic** VAD you can tune two sensitivity parameters:

| Parameter | Options | Effect |
|-----------|---------|--------|
| `start_of_speech_sensitivity` | `START_SENSITIVITY_LOW` / `HIGH` | How easily VAD triggers on speech start |
| `end_of_speech_sensitivity` | `END_SENSITIVITY_LOW` / `HIGH` | How quickly VAD decides the user has stopped |

**HIGH sensitivity** → reacts quickly; good for quiet environments  
**LOW sensitivity** → more noise-tolerant; good for noisy environments (cafes, call centers)

The cell below shows how to configure high sensitivity on both ends.

In [ ]:
async def demo_sensitivity(start_sens, end_sens, label: str):
    """Run a single turn with the given VAD sensitivity settings."""

    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        realtime_input_config=types.RealtimeInputConfig(
            automatic_activity_detection=types.AutomaticActivityDetection(
                start_of_speech_sensitivity=start_sens,
                end_of_speech_sensitivity=end_sens,
                # disabled is NOT set here — we keep automatic VAD ON
            )
        ),
        output_audio_transcription=types.AudioTranscriptionConfig(),
        system_instruction="You are a helpful assistant. Reply in one sentence.",
    )

    async with client.aio.live.connect(model=MODEL, config=config) as session:
        # Send text so the demo runs without a microphone
        await session.send_realtime_input(text="Tell me one interesting fact about dolphins.")
        result = await collect_response(session, label=label)

    return result


# ── High sensitivity (reactive) ────────────────────────────────────────────
print("=== HIGH start / HIGH end sensitivity ===")
r_high = asyncio.run(
    demo_sensitivity(
        types.StartSensitivity.START_SENSITIVITY_HIGH,
        types.EndSensitivity.END_SENSITIVITY_HIGH,
        label="HIGH/HIGH",
    )
)
print("Transcript:", r_high["transcript"])
if r_high["audio"]:
    display(play_pcm(r_high["audio"]))

print()

# ── Low sensitivity (noise-tolerant) ──────────────────────────────────────
print("=== LOW start / LOW end sensitivity ===")
r_low = asyncio.run(
    demo_sensitivity(
        types.StartSensitivity.START_SENSITIVITY_LOW,
        types.EndSensitivity.END_SENSITIVITY_LOW,
        label="LOW/LOW",
    )
)
print("Transcript:", r_low["transcript"])
if r_low["audio"]:
    display(play_pcm(r_low["audio"]))

### When to use which setting?

```
Environment          start_sensitivity    end_sensitivity
──────────────────   ─────────────────    ───────────────
Quiet room / lab     HIGH                 HIGH    (snap response)
Open-plan office     LOW                  HIGH    (filter noise, end quickly)
Call center / VOIP   LOW                  LOW     (tolerant of background)
Push-to-talk device  N/A (use manual)     N/A
```

---
## Demo 4 — Proactive Speaking (Model Speaks First)

A **proactive** agent greets the user immediately without waiting for input.
This is useful for:
- IVR systems that open with "Thank you for calling..."
- Kiosks that say "Welcome! How can I help you today?"
- Onboarding flows that guide the user step by step

### Technique
The system prompt instructs the model to speak immediately on connection.
We then send a **single space** as a minimal trigger so the model has something
to respond to — the space itself is semantically empty.

```python
await session.send_realtime_input(text=" ")
```

The model's system prompt drives the response, not the trigger text.

In [ ]:
PROACTIVE_SYSTEM_PROMPT = """\
You are a proactive customer service assistant for AcmeCorp.
As soon as the session starts, immediately introduce yourself by name ("Hi, I'm Aria"),
welcome the user warmly, and ask what they need help with today.
Keep the greeting under 3 sentences.
"""


async def demo_proactive_speaking():
    """Model speaks first — proactive greeting pattern."""

    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        output_audio_transcription=types.AudioTranscriptionConfig(),
        system_instruction=PROACTIVE_SYSTEM_PROMPT,
    )

    async with client.aio.live.connect(model=MODEL, config=config) as session:
        print("Session open — sending trigger...")

        # Single space: semantically empty, but gives the model something to react to.
        # The system prompt is what drives the actual greeting.
        await session.send_realtime_input(text=" ")

        print("Trigger sent — waiting for model greeting...")
        result = await collect_response(session, label="Proactive")

    return result


result4 = asyncio.run(demo_proactive_speaking())
print("\n=== Model greeting ===")
print(result4["transcript"] or "(audio only — play below)")

if result4["audio"]:
    print(f"\nAudio: {len(result4['audio']):,} bytes")
    display(play_pcm(result4["audio"]))

In [ ]:
# ── Extended proactive demo: multi-turn ───────────────────────────────────
# After the greeting, the user replies and the conversation continues.

PROACTIVE_PROMPT_V2 = """\
You are a friendly airport information kiosk assistant.
On session start, immediately say: 'Welcome to Changi Airport! I can help with
flight info, directions, or lounge access. What do you need?'
Keep every reply under 2 sentences.
"""


async def demo_proactive_multiturn():
    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        output_audio_transcription=types.AudioTranscriptionConfig(),
        system_instruction=PROACTIVE_PROMPT_V2,
    )

    turns = []

    async with client.aio.live.connect(model=MODEL, config=config) as session:
        # ── Turn 0: proactive greeting ─────────────────────────────────────
        await session.send_realtime_input(text=" ")
        r0 = await collect_response(session, label="Greeting")
        turns.append(("[kiosk]", r0["transcript"]))

        # ── Turn 1: user asks a question ───────────────────────────────────
        await session.send_realtime_input(text="Where is Terminal 3 lounge?")
        r1 = await collect_response(session, label="T1")
        turns.append(("[user]", "Where is Terminal 3 lounge?"))
        turns.append(("[kiosk]", r1["transcript"]))

        # ── Turn 2: follow-up ──────────────────────────────────────────────
        await session.send_realtime_input(text="How do I get there from Terminal 1?")
        r2 = await collect_response(session, label="T2")
        turns.append(("[user]", "How do I get there from Terminal 1?"))
        turns.append(("[kiosk]", r2["transcript"]))

        audio_samples = [r0["audio"], r1["audio"], r2["audio"]]

    return turns, audio_samples


turns, audio_list = asyncio.run(demo_proactive_multiturn())

print("\n=== Conversation ===")
for speaker, text in turns:
    print(f"{speaker}: {text}")

print("\n=== Kiosk audio (turn 1) ===")
if audio_list[0]:
    display(play_pcm(audio_list[0]))

---
## Key Takeaways

| Concept | Key Point |
|---------|----------|
| **Automatic VAD** | Default; server decides when speech ends; zero client code needed |
| **Manual VAD** | Set `disabled=True`; send `ActivityStart` / `ActivityEnd` explicitly |
| **turn_coverage** | `TURN_INCLUDES_ONLY_ACTIVITY` — only bracketed audio is sent to the model |
| **Sensitivity** | `HIGH` = reactive (quiet rooms), `LOW` = tolerant (noisy environments) |
| **Proactive speaking** | System prompt + single-space trigger → model greets first |
| **send_realtime_input** | The correct API for text, audio, ActivityStart/End — never `send_client_content` |

### Next Steps
- **Notebook 07** — Affective Dialog: tone and style adaptation  
- **Notebook 08** — Google Search grounding and session management

### Reference Links
- [Gemini Live API docs](https://ai.google.dev/gemini-api/docs/live)
- [google-genai Python SDK](https://github.com/googleapis/python-genai)